## Spectrogram Generation Pipeline

Build spectrogram PNGs from clipped WAV files using small utility steps so each stage is easy to inspect.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "config.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "config.py").exists():
    raise FileNotFoundError(f"Couldn't find src/config.py from {Path.cwd()}")

sys.path.insert(0, str(repo_root))

from src.config import CONFIG
from src.dataset.utils.spectrograms import (
    build_spectrogram_index_dataframe,
    collect_capped_non_bird_rows,
    collect_species_clip_rows,
    combine_and_shuffle_clip_rows,
    compute_class_counts,
    generate_spectrogram_pngs_from_index,
    resolve_spectrogram_build_paths,
    write_spectrogram_index_and_errors,
)
from src.dataset.utils.splits import build_default_splits_for_spectrograms_v1


### 1) Resolve Paths and Config


In [2]:
spec_cfg = CONFIG.spectrogram
train_shape_hw = spec_cfg.train_out_shape_hw

paths = resolve_spectrogram_build_paths(
    out_name="spectrograms_v1",
    train_shape_hw=train_shape_hw,
)

print("DATA_DIR:", paths.data_dir)
print("SPECIES_ROOT:", paths.species_root)
print("NON_BIRD_ROOT:", paths.non_bird_root)
print("FULL_DIR:", paths.full_dir)
print("TRAIN_DIR:", paths.train_dir)


DATA_DIR: bird_data
SPECIES_ROOT: bird_data/clips/species
NON_BIRD_ROOT: bird_data/clips/non_bird
FULL_DIR: bird_data/spectrograms_v1/full
TRAIN_DIR: bird_data/spectrograms_v1/train_64x128


### 2) Collect Clip Rows


In [3]:
non_bird_cap = 3000
seed = CONFIG.preprocessing.seed

species_rows = collect_species_clip_rows(paths.species_root)
non_bird_rows = collect_capped_non_bird_rows(
    paths.non_bird_root,
    non_bird_cap=non_bird_cap,
    seed=seed,
)
clip_rows = combine_and_shuffle_clip_rows(species_rows, non_bird_rows, seed=seed)

print("Species rows:", len(species_rows))
print("Non-bird rows:", len(non_bird_rows))
print("Total rows:", len(clip_rows))
clip_rows[:3]


RuntimeError: No wav files found. Check bird_data/clips/ layout.

### 3) Build Index DataFrame with Output Paths


In [ ]:
index_df = build_spectrogram_index_dataframe(
    clip_rows,
    full_dir=paths.full_dir,
    train_dir=paths.train_dir,
)

print(index_df.shape)
index_df.head()


### 4) Generate Spectrogram PNGs


In [ ]:
errors = generate_spectrogram_pngs_from_index(
    index_df,
    spec_cfg=spec_cfg,
    save_full=True,
    overwrite=False,
)

print("Generation errors:", len(errors))
if errors:
    errors[:5]


### 5) Write Index and Plot Class Counts


In [ ]:
index_csv = write_spectrogram_index_and_errors(
    index_df,
    out_root=paths.out_root,
    train_shape_hw=train_shape_hw,
    errors=errors,
)

print("Wrote index:", index_csv)

class_counts = compute_class_counts(index_df)
print(class_counts)

plt.figure(figsize=(12, 4))
class_counts.plot(kind="bar")
plt.title("Class counts (species + non_bird)")
plt.ylabel("count")
plt.tight_layout()
plt.show()


### 6) Build Splits


In [ ]:
split_paths = build_default_splits_for_spectrograms_v1()
split_paths
